In [1]:
import os
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun

from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_ollama import OllamaEmbeddings

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool

from langchain_classic.agents import create_tool_calling_agent, AgentExecutor


C:\Users\Leander Joseph\AppData\Local\Temp\ipykernel_196\415685400.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
wiki_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=wiki_wrapper)

wiki.name

'wikipedia'

In [3]:
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=arxiv_wrapper)

arxiv.name

'arxiv'

In [4]:
embedding = OllamaEmbeddings(model="mxbai-embed-large:latest")
embedding

OllamaEmbeddings(model='mxbai-embed-large:latest', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [5]:
loader = WebBaseLoader(web_path="https://docs.smith.langchain.com/")
doc = loader.load()

chunks = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap=250).split_documents(doc)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

retriever = vector_store.as_retriever()

In [6]:
retriever_tool = create_retriever_tool(retriever=retriever, name="Retriever_tool", description="Search any information about Langsmith")

In [7]:
tools = [wiki, arxiv, retriever_tool]

In [8]:
llm = ChatOllama(model="llama3.1")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with search capabilities. Use tools when necessary."),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

prompt                         

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='

In [9]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMes

In [10]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: message_formatter(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMe

In [12]:
response = agent_executor.invoke({"input":"What is GenAI?"})



> Entering new AgentExecutor chain...

Invoking: `Retriever_tool` with `{'query': 'GenAI'}`


Copy the key and save it securely.Once your account and API key are ready, set up tracing:Set up tracingAdd tracing to your app in minutes with environment variables, framework integrations, or the SDK.Trace a RAG applicationFollow a step-by-step tutorial to instrument a retrieval-augmented generation app from start to finish.Investigate and monitorView tracesInspect threads and runs in the Messages, Turns, and Details views.Monitor performanceBuild dashboards and set alerts to track quality and catch issues early.Configure automationsAutomate workflows with rules, webhooks, and online evaluations.Collect feedbackAnnotate outputs and gather user feedback using queues or inline annotation.Find and fix failures with EngineAutomatically detect recurring issues in your traces, diagnose their root cause, and resolve them with LangSmith Engine.If your applications call models through the LLM Gatew

In [14]:
response['output']

'Based on the tool output, GenAI (short for General Artificial Intelligence) is not explicitly defined in the provided output. However, the output is related to LangSmith Observability, which is a tool for monitoring and analyzing the performance of LLM (Large Language Model) applications.\n\nConsidering the context, GenAI is likely a term referring to General Artificial Intelligence, which is a broad area of research and development aimed at creating intelligent machines that can perform any intellectual task that humans can. In the context of LangSmith Observability, GenAI might be related to the capabilities of the tool to analyze and understand the performance of LLM applications, which are a key aspect of General Artificial Intelligence.\n\nTherefore, a possible answer to the original question is:\n\nGenAI is likely a term referring to General Artificial Intelligence, which is a broad area of research and development aimed at creating intelligent machines that can perform any inte

In [ ]:
response = agent_executor.invoke({"input":"What does it mean when a cat bites the caretaker's bite gently?"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Cat behavior when biting caretaker gently'}`


No good Wikipedia Search Result was foundThat's a curious behavior from a cat.  Based on common knowledge, when a cat bites its caretaker gently, it may be a sign of affection or playfulness. Cats often use their mouths to express affection, and a gentle bite can be a way for them to show love and attachment to their caregiver. However, it's also possible that the cat is trying to initiate play or is simply acting on instinct. If the bites are too frequent or aggressive, it's best to consult a veterinarian or animal behaviorist to ensure the cat's behavior isn't a sign of any underlying issues.

> Finished chain.


In [ ]:
response['output']

"That's a curious behavior from a cat.  Based on common knowledge, when a cat bites its caretaker gently, it may be a sign of affection or playfulness. Cats often use their mouths to express affection, and a gentle bite can be a way for them to show love and attachment to their caregiver. However, it's also possible that the cat is trying to initiate play or is simply acting on instinct. If the bites are too frequent or aggressive, it's best to consult a veterinarian or animal behaviorist to ensure the cat's behavior isn't a sign of any underlying issues."